# Группа 1. «Свой или чужой»

Запустите ячейку целиком. Затем меняйте параметры в начале кода и повторяйте запуск. Главная задача — сравнить случайное разделение кадров и разделение по физическим объектам.


In [1]:
"""
Практикум группы 1.

Синтетическая демонстрация классификации «свой/чужой».
Главный вопрос: почему случайное разделение кадров может дать ложное качество?

Запуск:
    python3 "Группа 1 — свой или чужой.py"

Результаты сохраняются в папке outputs_images.
"""

from pathlib import Path
from datetime import datetime

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, recall_score
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


# ПАРАМЕТРЫ, КОТОРЫЕ МОЖНО МЕНЯТЬ
SEED = 42
OBJECTS_PER_CLASS = 10
FRAMES_PER_OBJECT = 14
IMAGE_SIZE = 32
NOISE_LEVEL = 12
TEST_SHARE = 0.30


RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = Path.cwd() / "outputs_images" / RUN_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def generate_dataset():
    rng = np.random.default_rng(SEED)
    images, labels, object_ids = [], [], []

    for label in (0, 1):
        for object_number in range(OBJECTS_PER_CLASS):
            object_id = f"class_{label}_object_{object_number:02d}"

            # У каждого физического объекта свой фон.
            # При случайном разделении соседние кадры одного объекта попадут
            # и в train, и в test. Модель сможет запомнить фон.
            background = rng.integers(20, 220, size=3)
            object_tint = rng.integers(180, 256, size=3)
            base_size = int(rng.integers(11, 18))

            for _ in range(FRAMES_PER_OBJECT):
                canvas = np.zeros((IMAGE_SIZE, IMAGE_SIZE, 3), dtype=np.uint8)
                canvas[:] = background
                image = Image.fromarray(canvas)
                draw = ImageDraw.Draw(image)

                shift_x = int(rng.integers(-4, 5))
                shift_y = int(rng.integers(-4, 5))
                size = max(8, base_size + int(rng.integers(-2, 3)))
                x0 = IMAGE_SIZE // 2 - size // 2 + shift_x
                y0 = IMAGE_SIZE // 2 - size // 2 + shift_y
                x1, y1 = x0 + size, y0 + size

                if label == 0:
                    # «Свой»: круг.
                    draw.ellipse((x0, y0, x1, y1), fill=tuple(object_tint.tolist()))
                else:
                    # «Чужой»: квадрат с небольшим поворотом не моделируем,
                    # чтобы эксперимент оставался быстрым и понятным.
                    draw.rectangle((x0, y0, x1, y1), fill=tuple(object_tint.tolist()))

                array = np.asarray(image).astype(np.int16)
                noise = rng.normal(0, NOISE_LEVEL, size=array.shape)
                array = np.clip(array + noise, 0, 255).astype(np.uint8)

                images.append(array)
                labels.append(label)
                object_ids.append(object_id)

    return np.stack(images), np.asarray(labels), np.asarray(object_ids)


def train_and_evaluate(name, images, labels, train_idx, test_idx):
    X = images.reshape(len(images), -1).astype(np.float32) / 255.0
    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=600, random_state=SEED),
    )
    model.fit(X[train_idx], labels[train_idx])
    prediction = model.predict(X[test_idx])
    result = {
        "split": name,
        "accuracy": round(float(accuracy_score(labels[test_idx], prediction)), 3),
        "recall_foreign": round(float(recall_score(labels[test_idx], prediction, pos_label=1)), 3),
        "test_images": int(len(test_idx)),
    }
    return model, prediction, result


images, labels, object_ids = generate_dataset()
indices = np.arange(len(images))

# Эксперимент 1: неправильное для проекта случайное разделение кадров.
random_train, random_test = train_test_split(
    indices,
    test_size=TEST_SHARE,
    random_state=SEED,
    stratify=labels,
)

# Эксперимент 2: честнее — физический объект целиком только в одной части.
splitter = GroupShuffleSplit(n_splits=1, test_size=TEST_SHARE, random_state=SEED)
group_train, group_test = next(splitter.split(indices, labels, groups=object_ids))

random_model, random_prediction, random_result = train_and_evaluate(
    "Случайно по кадрам", images, labels, random_train, random_test
)
group_model, group_prediction, group_result = train_and_evaluate(
    "По физическим объектам", images, labels, group_train, group_test
)

results = pd.DataFrame([random_result, group_result])
print("\nСРАВНЕНИЕ ДВУХ СПОСОБОВ РАЗДЕЛЕНИЯ")
print(results.to_string(index=False))
results.to_csv(OUTPUT_DIR / "metrics.csv", index=False)

# Проверяем пересечение объектов.
random_overlap = set(object_ids[random_train]) & set(object_ids[random_test])
group_overlap = set(object_ids[group_train]) & set(object_ids[group_test])
print("\nПересечение физических объектов:")
print("Случайное разделение:", len(random_overlap))
print("Разделение по объектам:", len(group_overlap))

# Показываем примеры.
fig, axes = plt.subplots(3, 6, figsize=(12, 6))
for ax, idx in zip(axes.flat, indices[:18]):
    ax.imshow(images[idx])
    ax.set_title(f"{'свой' if labels[idx] == 0 else 'чужой'}\n{object_ids[idx].split('_')[-1]}")
    ax.axis("off")
fig.suptitle("Примеры синтетических кадров")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "examples.png", dpi=150)
plt.close(fig)

# Матрицы ошибок.
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
ConfusionMatrixDisplay.from_predictions(
    labels[random_test],
    random_prediction,
    display_labels=["свой", "чужой"],
    ax=axes[0],
    colorbar=False,
)
axes[0].set_title("Случайно по кадрам")
ConfusionMatrixDisplay.from_predictions(
    labels[group_test],
    group_prediction,
    display_labels=["свой", "чужой"],
    ax=axes[1],
    colorbar=False,
)
axes[1].set_title("По физическим объектам")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "confusion_matrices.png", dpi=150)
plt.close(fig)

# Ошибки честного эксперимента.
errors = group_test[group_prediction != labels[group_test]]
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for ax in axes.flat:
    ax.axis("off")
for ax, idx in zip(axes.flat, errors[:10]):
    ax.imshow(images[idx])
    ax.set_title(
        f"истина: {'свой' if labels[idx] == 0 else 'чужой'}\n"
        f"объект: {object_ids[idx].split('_')[-1]}"
    )
    ax.axis("off")
fig.suptitle("Ошибки при проверке на новых физических объектах")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "errors.png", dpi=150)
plt.close(fig)

print("\nЗАДАНИЕ ГРУППЫ")
print("1. Какое разделение показало более высокий результат и почему?")
print("2. Почему высокий результат случайного разделения может быть ложным?")
print("3. Измените OBJECTS_PER_CLASS и FRAMES_PER_OBJECT. Что важнее?")
print("4. Увеличьте NOISE_LEVEL. Как изменился recall класса «чужой»?")
print("5. Предложите первую реальную модель: небольшая CNN или transfer learning.")
print("6. Запишите, какие фотографии нужно собрать для реального проекта.")
print(f"\nФайлы результата: {OUTPUT_DIR.resolve()}")



СРАВНЕНИЕ ДВУХ СПОСОБОВ РАЗДЕЛЕНИЯ
                 split  accuracy  recall_foreign  test_images
    Случайно по кадрам     0.833           0.810           84
По физическим объектам     0.667           0.643           84

Пересечение физических объектов:
Случайное разделение: 20
Разделение по объектам: 0

ЗАДАНИЕ ГРУППЫ
1. Какое разделение показало более высокий результат и почему?
2. Почему высокий результат случайного разделения может быть ложным?
3. Измените OBJECTS_PER_CLASS и FRAMES_PER_OBJECT. Что важнее?
4. Увеличьте NOISE_LEVEL. Как изменился recall класса «чужой»?
5. Предложите первую реальную модель: небольшая CNN или transfer learning.
6. Запишите, какие фотографии нужно собрать для реального проекта.

Файлы результата: /Users/dkh/Documents/Аспирантура/Педагогическая_практика/Информационные_системы_в_робототехнике_2026/Практика/outputs_images/20261015_145136


## Что сдаёт группа

1. Таблицу результатов.
2. Один основной график.
3. Выбранную первую модель.
4. Главный риск данных.
5. Следующий эксперимент.
